In [2]:
# =============================================================
# NOTEBOOK 04 – CLUSTERING (GOM CỤM)
# Môn: Học Máy MAT3533
# Đề tài: Hồi quy – IBM HR Analytics Employee Attrition
# Thực hiện: Phan Thanh Lâm
# Tuần: 2 | Ngày T2 – T4
# =============================================================

# ─── MỤC TIÊU ───────────────────────────────────────────────
# 1. K-Means Clustering (Elbow method + Silhouette)
# 2. GMM (Gaussian Mixture Model) với AIC/BIC
# 3. So sánh K-Means vs GMM
# 4. Visualize clusters trong PCA space
# 5. Phân tích đặc trưng của từng cluster

# ─── IMPORTS ───────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA
import os
import warnings
warnings.filterwarnings('ignore')

# Style
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 150,
})

# Màu sắc
COLORS = {
    'purple': '#534AB7',
    'teal': '#1D9E75',
    'coral': '#E24B4A',
    'amber': '#EF9F27',
    'pink': '#D4537E',
    'blue': '#1f77b4',
    'green': '#2ca02c',
    'orange': '#ff7f0e',
    'cluster': ['#E24B4A', '#1D9E75', '#534AB7', '#EF9F27', '#D4537E', '#1f77b4', '#2ca02c', '#9467bd']
}

# ─── PATHS ─────────────────────────────────────────────────
SCALED_PATH = '../data/processed/df_scaled.csv'
PCA_PATH = '../data/processed/df_pca.csv'
RESULTS_PATH = '../results'
os.makedirs(RESULTS_PATH, exist_ok=True)
os.makedirs('../outputs/figures', exist_ok=True)

print("=" * 60)
print("NOTEBOOK 04 – CLUSTERING (GOM CỤM)")
print("=" * 60)

# ═══════════════════════════════════════════════════════════
# BƯỚC 1 – LOAD DATA
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("BƯỚC 1 – LOAD DATA")
print("=" * 60)

# Load scaled data (bỏ target để clustering)
df_scaled = pd.read_csv(SCALED_PATH)
print(f"✓ Loaded df_scaled: {df_scaled.shape}")

# Load PCA data cho visualization
df_pca = pd.read_csv(PCA_PATH)
print(f"✓ Loaded df_pca: {df_pca.shape}")

# Chuẩn bị data cho clustering (bỏ target và cột không cần thiết)
TARGET = 'MonthlyIncome'
exclude_cols = [TARGET, 'Attrition']
X = df_scaled.drop(columns=exclude_cols, errors='ignore')
X_pca_viz = df_pca[[f'PC{i+1}' for i in range(10)]].values

# Lấy thông tin cho analysis
y_true_income = df_scaled[TARGET].values
job_level = df_scaled['JobLevel'].values if 'JobLevel' in df_scaled.columns else None

print(f"\n✓ Data cho clustering: {X.shape[1]} features, {X.shape[0]} samples")
print(f"✓ PCA data cho viz: {X_pca_viz.shape[1]} components")

# ═══════════════════════════════════════════════════════════
# BƯỚC 2 – K-MEANS: ELBOW METHOD & SILHOUETTE
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("BƯỚC 2 – K-MEANS CLUSTERING")
print("=" * 60)

K_range = range(2, 9)
inertias = []
silhouettes = []
davies_bouldins = []

print(f"\nĐánh giá K-Means với K = 2 → 8:")
print("-" * 50)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X)
    
    inertias.append(kmeans.inertia_)
    sil = silhouette_score(X, labels)
    silhouettes.append(sil)
    db = davies_bouldin_score(X, labels)
    davies_bouldins.append(db)
    
    print(f"  K={k}: Inertia={kmeans.inertia_:>12,.0f} | Silhouette={sil:.4f} | Davies-Bouldin={db:.4f}")

# Tìm K tối ưu
best_k_silhouette = K_range[np.argmax(silhouettes)]
best_k_db = K_range[np.argmin(davies_bouldins)]

print(f"\n✓ K tối ưu theo Silhouette: K={best_k_silhouette} (score={max(silhouettes):.4f})")
print(f"✓ K tối ưu theo Davies-Bouldin: K={best_k_db} (score={min(davies_bouldins):.4f})")

# Chọn K tối ưu (ưu tiên Silhouette)
optimal_k = best_k_silhouette
print(f"\n👉 Chọn K={optimal_k} cho phân tích tiếp theo")

# Hình 11: Elbow + Silhouette
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Hình 11: K-Means - Tìm số cụm tối ưu', fontsize=13, fontweight='bold')

# Elbow plot
axes[0].plot(K_range, inertias, 'o-', color=COLORS['purple'], linewidth=2, markersize=8)
axes[0].axvline(x=optimal_k, color='red', linestyle='--', label=f'K={optimal_k} (chọn)')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].set_title('Elbow Method')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Silhouette plot
axes[1].bar(K_range, silhouettes, color=COLORS['teal'], alpha=0.7)
axes[1].axhline(y=max(silhouettes), color='red', linestyle='--', label=f'Max={max(silhouettes):.3f}')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../outputs/figures/fig11_kmeans_elbow_silhouette.png', bbox_inches='tight', dpi=150)
plt.close()
print("✓ Saved: outputs/figures/fig11_kmeans_elbow_silhouette.png")

# ═══════════════════════════════════════════════════════════
# BƯỚC 3 – K-MEANS VỚI K TỐI ƯU
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print(f"BƯỚC 3 – K-MEANS (K={optimal_k})")
print("=" * 60)

kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
cluster_labels = kmeans_final.fit_predict(X)

# Thống kê từng cluster
print(f"\nPhân bố samples theo cluster:")
cluster_counts = pd.Series(cluster_labels).value_counts().sort_index()
for c in range(optimal_k):
    count = cluster_counts[c]
    pct = count / len(cluster_labels) * 100
    print(f"  Cluster {c}: {count} samples ({pct:.1f}%)")

# Phân tích MonthlyIncome theo cluster
print(f"\nMonthlyIncome theo cluster:")
income_by_cluster = []
for c in range(optimal_k):
    mask = cluster_labels == c
    income_mean = y_true_income[mask].mean()
    income_std = y_true_income[mask].std()
    print(f"  Cluster {c}: Mean={income_mean:>8,.0f} | Std={income_std:>7,.0f} USD")
    income_by_cluster.append({'cluster': c, 'mean_income': income_mean, 'std_income': income_std})

# Hình 12: K-Means clusters trong PCA space
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(f'Hình 12: K-Means Clusters (K={optimal_k})', fontsize=13, fontweight='bold')

# Màu sắc cho clusters
cluster_colors = [COLORS['cluster'][c % len(COLORS['cluster'])] for c in range(optimal_k)]

# PCA 2D
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X)

scatter1 = axes[0].scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=cluster_labels, cmap='tab10', alpha=0.5, s=15)
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].set_title('K-Means Clusters (PCA 2D)')
axes[0].grid(True, alpha=0.3)

# t-SNE 2D (nếu có file)
try:
    df_tsne = pd.read_csv('../data/processed/df_tsne.csv')
    X_tsne = df_tsne[['tSNE_1', 'tSNE_2']].values
    scatter2 = axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=cluster_labels, cmap='tab10', alpha=0.5, s=15)
    axes[1].set_xlabel('t-SNE 1')
    axes[1].set_ylabel('t-SNE 2')
    axes[1].set_title('K-Means Clusters (t-SNE 2D)')
    axes[1].grid(True, alpha=0.3)
except:
    axes[1].text(0.5, 0.5, 't-SNE data not available', ha='center', va='center', transform=axes[1].transAxes)
    axes[1].set_title('t-SNE Visualization (not available)')

plt.colorbar(scatter1, ax=axes[0], label='Cluster')
plt.tight_layout()
plt.savefig('../outputs/figures/fig12_kmeans_clusters.png', bbox_inches='tight', dpi=150)
plt.close()
print("✓ Saved: outputs/figures/fig12_kmeans_clusters.png")

# ═══════════════════════════════════════════════════════════
# BƯỚC 4 – GMM (GAUSSIAN MIXTURE MODEL)
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("BƯỚC 4 – GMM (Gaussian Mixture Model)")
print("=" * 60)

aics = []
bics = []
gmm_silhouettes = []

print(f"\nĐánh giá GMM với K = 2 → 8:")
print("-" * 50)

for k in K_range:
    gmm = GaussianMixture(n_components=k, random_state=42, max_iter=500)
    gmm_labels = gmm.fit_predict(X)
    
    aics.append(gmm.aic(X))
    bics.append(gmm.bic(X))
    sil = silhouette_score(X, gmm_labels)
    gmm_silhouettes.append(sil)
    
    print(f"  K={k}: AIC={gmm.aic(X):>12,.0f} | BIC={gmm.bic(X):>12,.0f} | Silhouette={sil:.4f}")

# Tìm K tối ưu cho GMM (theo BIC)
best_k_bic = K_range[np.argmin(bics)]
best_k_aic = K_range[np.argmin(aics)]

print(f"\n✓ K tối ưu theo BIC: K={best_k_bic}")
print(f"✓ K tối ưu theo AIC: K={best_k_aic}")

# Chọn K cho GMM (ưu tiên BIC)
optimal_k_gmm = best_k_bic
print(f"\n👉 Chọn K={optimal_k_gmm} cho GMM")

# Train GMM với K tối ưu
gmm_final = GaussianMixture(n_components=optimal_k_gmm, random_state=42, max_iter=1000)
gmm_labels = gmm_final.fit_predict(X)
gmm_proba = gmm_final.predict_proba(X)

print(f"\nPhân bố samples theo GMM clusters:")
gmm_counts = pd.Series(gmm_labels).value_counts().sort_index()
for c in range(optimal_k_gmm):
    count = gmm_counts[c] if c in gmm_counts else 0
    pct = count / len(gmm_labels) * 100
    print(f"  GMM Cluster {c}: {count} samples ({pct:.1f}%)")

# Hình 13: GMM clusters
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(f'Hình 13: GMM Clusters (K={optimal_k_gmm})', fontsize=13, fontweight='bold')

scatter1 = axes[0].scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=gmm_labels, cmap='tab10', alpha=0.5, s=15)
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].set_title('GMM Clusters (PCA 2D)')
axes[0].grid(True, alpha=0.3)

try:
    scatter2 = axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=gmm_labels, cmap='tab10', alpha=0.5, s=15)
    axes[1].set_xlabel('t-SNE 1')
    axes[1].set_ylabel('t-SNE 2')
    axes[1].set_title('GMM Clusters (t-SNE 2D)')
    axes[1].grid(True, alpha=0.3)
except:
    axes[1].text(0.5, 0.5, 't-SNE data not available', ha='center', va='center', transform=axes[1].transAxes)

plt.colorbar(scatter1, ax=axes[0], label='GMM Cluster')
plt.tight_layout()
plt.savefig('../outputs/figures/fig13_gmm_clusters.png', bbox_inches='tight', dpi=150)
plt.close()
print("✓ Saved: outputs/figures/fig13_gmm_clusters.png")

# ═══════════════════════════════════════════════════════════
# BƯỚC 5 – SO SÁNH K-MEANS vs GMM
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("BƯỚC 5 – SO SÁNH K-MEANS vs GMM")
print("=" * 60)

comparison_data = []
for k in K_range:
    # K-Means
    kmeans_temp = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans_labels = kmeans_temp.fit_predict(X)
    kmeans_sil = silhouette_score(X, kmeans_labels)
    
    # GMM
    gmm_temp = GaussianMixture(n_components=k, random_state=42)
    gmm_labels = gmm_temp.fit_predict(X)
    gmm_sil = silhouette_score(X, gmm_labels)
    
    comparison_data.append({
        'K': k,
        'KMeans_Silhouette': round(kmeans_sil, 4),
        'GMM_Silhouette': round(gmm_sil, 4),
        'Difference': round(gmm_sil - kmeans_sil, 4)
    })

comparison_df = pd.DataFrame(comparison_data)
print("\n" + comparison_df.to_string(index=False))

# Hình 14: So sánh Silhouette
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(comparison_df['K'], comparison_df['KMeans_Silhouette'], 'o-', 
        label='K-Means', color=COLORS['purple'], linewidth=2, markersize=8)
ax.plot(comparison_df['K'], comparison_df['GMM_Silhouette'], 's-', 
        label='GMM', color=COLORS['teal'], linewidth=2, markersize=8)
ax.set_xlabel('Number of Clusters (K)')
ax.set_ylabel('Silhouette Score')
ax.set_title('Hình 14: So sánh Silhouette Score - K-Means vs GMM', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/fig14_kmeans_vs_gmm.png', bbox_inches='tight', dpi=150)
plt.close()
print("✓ Saved: outputs/figures/fig14_kmeans_vs_gmm.png")

# ═══════════════════════════════════════════════════════════
# BƯỚC 6 – PHÂN TÍCH ĐẶC TRƯNG CLUSTER
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("BƯỚC 6 – PHÂN TÍCH ĐẶC TRƯNG CÁC CLUSTER")
print("=" * 60)

# Sử dụng K-Means clusters để phân tích
df_analysis = df_scaled.copy()
df_analysis['Cluster'] = cluster_labels
df_analysis['MonthlyIncome'] = y_true_income

# Thống kê theo cluster
cluster_stats = df_analysis.groupby('Cluster').agg({
    'MonthlyIncome': ['mean', 'std', 'min', 'max'],
    'JobLevel': 'mean',
    'YearsAtCompany': 'mean',
    'TotalWorkingYears': 'mean'
}).round(2)

print("\n📊 Thống kê theo Cluster (K-Means):")
print(cluster_stats.to_string())

# Hình 15: Boxplot MonthlyIncome theo cluster
fig, ax = plt.subplots(figsize=(10, 6))
bp = df_analysis.boxplot(column='MonthlyIncome', by='Cluster', ax=ax, 
                          color=dict(boxes=COLORS['purple'], whiskers='black', medians='red'))
ax.set_title('Hình 15: Phân phối MonthlyIncome theo Cluster', fontweight='bold')
ax.set_xlabel('Cluster')
ax.set_ylabel('MonthlyIncome (USD)')
plt.suptitle('')
plt.tight_layout()
plt.savefig('../outputs/figures/fig15_income_by_cluster.png', bbox_inches='tight', dpi=150)
plt.close()
print("✓ Saved: outputs/figures/fig15_income_by_cluster.png")

# ═══════════════════════════════════════════════════════════
# BƯỚC 7 – LƯU KẾT QUẢ
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("BƯỚC 7 – LƯU KẾT QUẢ")
print("=" * 60)

# Lưu labels
cluster_results = pd.DataFrame({
    'Cluster_KMeans': cluster_labels,
    'Cluster_GMM': gmm_labels,
    'MonthlyIncome': y_true_income,
    'JobLevel': job_level if job_level is not None else np.nan
})
cluster_results.to_csv(f'{RESULTS_PATH}/clustering_results.csv', index=False)
print(f"✓ Saved: {RESULTS_PATH}/clustering_results.csv")

# Lưu metrics
clustering_metrics = {
    'kmeans': {
        'optimal_k': int(optimal_k),
        'silhouette_score': float(max(silhouettes)),
        'davies_bouldin_score': float(min(davies_bouldins)),
        'inertia': float(kmeans_final.inertia_)
    },
    'gmm': {
        'optimal_k': int(optimal_k_gmm),
        'silhouette_score': float(gmm_silhouettes[list(K_range).index(optimal_k_gmm)]),
        'best_bic': float(min(bics)),
        'best_aic': float(min(aics))
    },
    'comparison': comparison_data
}

import json
with open(f'{RESULTS_PATH}/clustering_metrics.json', 'w') as f:
    json.dump(clustering_metrics, f, indent=2)
print(f"✓ Saved: {RESULTS_PATH}/clustering_metrics.json")

# ═══════════════════════════════════════════════════════════
# TÓM TẮT CUỐI
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("📊 TÓM TẮT CLUSTERING")
print("=" * 60)
print(f"""
  K-Means:
    - Optimal K         : {optimal_k}
    - Silhouette score  : {max(silhouettes):.4f}
    - Davies-Bouldin    : {min(davies_bouldins):.4f}
  
  GMM:
    - Optimal K (BIC)   : {optimal_k_gmm}
    - Optimal K (AIC)   : {best_k_aic}
    - Silhouette score  : {gmm_silhouettes[list(K_range).index(optimal_k_gmm)]:.4f}
  
  So sánh:
    - Phương pháp tốt hơn: {'GMM' if max(gmm_silhouettes) > max(silhouettes) else 'K-Means'}
    - Chênh lệch Silhouette: {abs(max(gmm_silhouettes) - max(silhouettes)):.4f}
  
  Output files:
    📁 results/clustering_results.csv
    📁 results/clustering_metrics.json
    📁 outputs/figures/fig11_kmeans_elbow_silhouette.png
    📁 outputs/figures/fig12_kmeans_clusters.png
    📁 outputs/figures/fig13_gmm_clusters.png
    📁 outputs/figures/fig14_kmeans_vs_gmm.png
    📁 outputs/figures/fig15_income_by_cluster.png
""")
print("=" * 60)
print("✅ notebook_04_clustering - HOÀN THÀNH!")

NOTEBOOK 04 – CLUSTERING (GOM CỤM)

BƯỚC 1 – LOAD DATA
✓ Loaded df_scaled: (1470, 48)
✓ Loaded df_pca: (1470, 12)

✓ Data cho clustering: 46 features, 1470 samples
✓ PCA data cho viz: 10 components

BƯỚC 2 – K-MEANS CLUSTERING

Đánh giá K-Means với K = 2 → 8:
--------------------------------------------------
  K=2: Inertia=      62,676 | Silhouette=0.0925 | Davies-Bouldin=3.3538
  K=3: Inertia=      58,850 | Silhouette=0.0840 | Davies-Bouldin=2.9480
  K=4: Inertia=      56,386 | Silhouette=0.0890 | Davies-Bouldin=2.7226
  K=5: Inertia=      53,411 | Silhouette=0.1068 | Davies-Bouldin=2.3670
  K=6: Inertia=      51,688 | Silhouette=0.0858 | Davies-Bouldin=2.6565
  K=7: Inertia=      50,130 | Silhouette=0.1012 | Davies-Bouldin=2.4206
  K=8: Inertia=      48,156 | Silhouette=0.1134 | Davies-Bouldin=2.3837

✓ K tối ưu theo Silhouette: K=8 (score=0.1134)
✓ K tối ưu theo Davies-Bouldin: K=5 (score=2.3670)

👉 Chọn K=8 cho phân tích tiếp theo
✓ Saved: outputs/figures/fig11_kmeans_elbow_silhou